# Agent 第 05 课：记忆（短期 vs 长期）

目前我们的 agent **只有一个对话里**记忆——`messages` 数组。两个问题：

1. 对话一长，上下文炸。20 步工具调用的 agent，最后一次调用的 prompt 能有 30k+ token
2. **跨会话**它什么都不记得。用户昨天告诉它"我叫 Alice，住美国"，今天又要重说一遍

两个问题对应两种记忆：

| 类型 | 存在哪 | 解决什么 | 实现 |
|---|---|---|---|
| **短期记忆** | 当前 `messages` | 上下文不炸 | 滑动窗口 / 摘要压缩 |
| **长期记忆** | 外部存储 | 跨会话记住事 | 向量库（直接复用 RAG 技术） |

这一课把两种都跑一遍。

In [ ]:
import json, re, numpy as np
from openai import OpenAI
client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
chat_model = next((m for m in ['qwen/qwen3.6-35b-a3b','qwen3.6-35b-a3b','qwen3.5-35b-a3b'] if m in model_ids), model_ids[0])
embedding_model = next((m for m in model_ids if 'embed' in m.lower()), None)
print('chat =', chat_model, '| embed =', embedding_model)

## 1. 短期记忆策略 A：滑动窗口

最简单——只保留最近 N 条消息，老的扔掉。

```python
def sliding_window(messages, keep_last=12):
    system = [m for m in messages if m['role']=='system']
    rest = [m for m in messages if m['role']!='system']
    return system + rest[-keep_last:]
```

**优点**：实现 5 行，零调用成本。
**缺点**：老的关键信息会被整条丢掉。用户 5 轮前说过"我叫 Alice"，模型记不住。

**什么时候用**：任务本身就是局部的（比如客服问答每轮相对独立），或者早期原型。

## 2. 短期记忆策略 B：摘要缓冲（Summary Buffer）

好得多的方案——**老消息不丢，压缩成摘要保留**。

```
[system]
[summary of turns 1..8]   ← 由 LLM 压缩出来的段落
[turn 9]
[turn 10]
[turn 11]
[turn 12 ← current]
```

触发条件通常是 token 数超过阈值。实现：每次超过时，让一个便宜的 LLM 把"到目前为止的对话"总结成 200 字塞回去。

In [ ]:
def approx_tokens(text): return max(1, len(text) // 3)  # 粗略估算

def total_tokens(messages):
    return sum(approx_tokens(m['content']) for m in messages)

def summarize(messages):
    """把非 system 的历史压缩成一段摘要。"""
    convo = '\n'.join([f"{m['role']}: {m['content']}" for m in messages if m['role']!='system'])
    r = client.chat.completions.create(
        model=chat_model, temperature=0,
        messages=[{'role':'system','content':'You summarize conversations in under 150 words. Preserve: user\'s stated facts, decisions made, any pending tasks. Be concrete.'},
                  {'role':'user','content':convo}])
    return r.choices[0].message.content.strip()

def summary_buffer(messages, budget_tokens=2000, keep_recent=4):
    """超过 budget 就把前面的压缩掉，只保留最近 keep_recent 轮。"""
    if total_tokens(messages) <= budget_tokens:
        return messages
    system = [m for m in messages if m['role']=='system']
    rest = [m for m in messages if m['role']!='system']
    to_compress = rest[:-keep_recent] if len(rest)>keep_recent else []
    recent = rest[-keep_recent:]
    if not to_compress: return messages
    summary = summarize(to_compress)
    return system + [{'role':'system','content':f'Summary of earlier conversation:\n{summary}'}] + recent

跑一个多轮对话，看压缩前后 token 数。

In [ ]:
fake_history = [
    {'role':'system','content':'You are a helpful assistant.'},
    {'role':'user','content':'Hi! I am Alice, I live in San Francisco and work as a data scientist at a small startup. I love hiking.'},
    {'role':'assistant','content':'Nice to meet you, Alice! What can I help with?'},
    {'role':'user','content':'I am trying to decide if I should buy a GPU for my home PC or use cloud GPUs.'},
    {'role':'assistant','content':'Both work. Cloud wins on flexibility and upfront cost; local wins on unit cost if you use it >4h/day.'},
    {'role':'user','content':'I only train small models, maybe 5h per week.'},
    {'role':'assistant','content':'Then cloud (e.g. RunPod, Lambda) is probably cheaper for you.'},
    {'role':'user','content':'Great. By the way, can you remember that my preferred cloud region is us-west-2?'},
    {'role':'assistant','content':'Noted: you prefer us-west-2.'},
] * 3  # 人为放大

print('before tokens ~', total_tokens(fake_history))
compressed = summary_buffer(fake_history, budget_tokens=500, keep_recent=4)
print('after tokens  ~', total_tokens(compressed))
print('\n--- compressed messages ---')
for m in compressed: print(f"[{m['role']}] {m['content'][:120]}...")

## 3. 长期记忆：向量存储 + 按相关性召回

这里你会发现：**长期记忆就是 RAG**。每次用户说了重要事实，存一条；下次开对话，把当前问题 embed 一下，去向量库检索相关的记忆条，塞进 system prompt。

我们用内存里的简易向量库（RAG 14 课那个 numpy 版），跑通概念。

In [ ]:
def embed(texts):
    r = client.embeddings.create(model=embedding_model, input=texts)
    return np.array([x.embedding for x in r.data], dtype=np.float32)
def l2n(m):
    n=np.linalg.norm(m,axis=1,keepdims=True); return m/np.clip(n,1e-12,None)

class MemoryStore:
    def __init__(self):
        self.facts = []        # List[str]
        self.vecs = None       # np.array [N, D]
    def add(self, fact):
        self.facts.append(fact)
        v = l2n(embed([fact]))
        self.vecs = v if self.vecs is None else np.vstack([self.vecs, v])
    def recall(self, query, k=3, threshold=0.3):
        if self.vecs is None: return []
        q = l2n(embed([query]))[0]
        sims = self.vecs @ q
        idx = np.argsort(sims)[::-1][:k]
        return [(self.facts[i], float(sims[i])) for i in idx if sims[i] > threshold]

mem = MemoryStore()
mem.add('User name is Alice.')
mem.add('Alice lives in San Francisco.')
mem.add('Alice prefers AWS region us-west-2.')
mem.add('Alice is a data scientist; she likes hiking.')
mem.add('Alice uses cloud GPUs about 5 hours a week.')

print('Q1 recall:', mem.recall('Which cloud region should I default to?'))
print()
print('Q2 recall:', mem.recall('What are my hobbies?'))

## 4. 一个带长期记忆的小 agent

每次请求前先 recall 相关记忆，塞进 system prompt。

**关键问题**：什么时候 **写入** 记忆？两种思路：
1. **显式**：提供一个 `remember(fact)` 工具，由模型自己决定何时存
2. **隐式**：对话结束后，后台用一个 "extractor" LLM 扫出用户说的事实自动存

这一课用思路 1——给 agent 一个 `remember` 工具，让它自己挑值得记的。

In [ ]:
TOOLS = [
    {'name':'remember',
     'description':'Save a durable fact about the user that will be useful in future conversations. Only save stable facts (preferences, name, location, role) — not ephemeral questions.',
     'parameters':{'type':'object','properties':{'fact':{'type':'string'}},'required':['fact']}},
    {'name':'recall',
     'description':'Recall facts about the user relevant to a query.',
     'parameters':{'type':'object','properties':{'query':{'type':'string'}},'required':['query']}},
]

def exec_tool(name, args, store):
    if name=='remember':
        store.add(args['fact']); return {'ok':True, 'saved':args['fact']}
    if name=='recall':
        return {'matches': store.recall(args['query'])}
    return {'error':f'unknown tool {name}'}

SYSTEM = f"""You are a personal assistant with persistent memory.

Tools:
{json.dumps(TOOLS, indent=2)}

At the start of each conversation, call `recall` with a short query describing the current topic to pull relevant facts.
When the user shares stable facts about themselves (name, location, preferences, role), save them with `remember`.
Do not remember ephemeral questions like 'what is the weather today'.

Protocol: <tool_call>...</tool_call> then wait; when done say 'Final Answer: ...'."""

TOOL_CALL_RE = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.S)
FINAL_RE = re.compile(r'Final Answer:\s*(.+)', re.S)

def run_agent_with_memory(user_msg, store, max_steps=6, verbose=True):
    messages=[{'role':'system','content':SYSTEM},{'role':'user','content':user_msg}]
    for step in range(1, max_steps+1):
        r = client.chat.completions.create(model=chat_model, temperature=0, messages=messages)
        t = r.choices[0].message.content
        messages.append({'role':'assistant','content':t})
        if verbose: print(f'--- step {step} ---'); print(t); print()
        mf = FINAL_RE.search(t)
        if mf: return mf.group(1).strip()
        mt = TOOL_CALL_RE.search(t)
        if mt:
            call = json.loads(mt.group(1))
            obs = exec_tool(call['name'], call.get('arguments') or {}, store)
            if verbose: print('>>>', obs, '\n')
            messages.append({'role':'user','content':f'<tool_response>\n{json.dumps(obs)}\n</tool_response>'})
        else:
            messages.append({'role':'user','content':'Please issue a <tool_call> or give a Final Answer.'})
    return None

## 5. 两次对话，看记忆起作用

In [ ]:
store = MemoryStore()
print('>>> SESSION 1')
run_agent_with_memory("Hi! I'm Alice, I'm a data scientist in San Francisco, and I mostly use AWS us-west-2.", store, verbose=False)
print('memory now:', store.facts)

In [ ]:
print('>>> SESSION 2 (new conversation, same store)')
ans = run_agent_with_memory('Which AWS region should I deploy my new service in?', store, verbose=True)
print('ANSWER:', ans)

## 6. 工程直觉

1. **短期记忆 ≠ 长期记忆**。别用一个机制解决两件事。短期是"当前上下文不炸"，长期是"跨会话知识"。
2. **摘要缓冲比滑动窗口几乎总是更好**——成本只是多一次 LLM 调用，但关键事实不丢。生产里默认上这个。
3. **长期记忆就是 RAG**。别被新词吓到，写入就是 upsert，读取就是检索。我们 RAG 21 课里关于向量库、多租户、权限过滤的经验**全部适用**。
4. **"何时写"比"怎么存"难**。记太多 → 召回噪声；记太少 → 没用。`remember` 工具的 description 必须明确"只记稳定事实"。
5. **记忆也要去重 / 过期**。用户改名字了，旧记忆要覆盖/标记 stale。生产里每条记忆通常带 `created_at` + `confidence` + `source_session_id`。
6. **隐私 ⚠️**：长期记忆会存用户个人信息，入库前要考虑 PII 脱敏、用户清除权（GDPR right-to-be-forgotten）。

---

下一课：**第 06 课 Planning（Plan-and-Execute）**——ReAct 是"边想边做"，有时候做之前先整体规划一遍效果更好。讲 Planner / Executor 的分离，以及什么时候该用哪个。